# Демо: замыкания и декораторы

Прокликай Shift+Enter каждую ячейку и посмотри, как функции живут как объекты, как замыкание запоминает переменные внешней области, и как декоратор оборачивает функцию, не меняя её код. В конце — три мини-задания.

## Часть 1. Функция — это обычный объект

В Python функцию можно положить в переменную, передать другой функции как аргумент, вернуть из функции, положить в список или словарь. Это называется «функция первого класса».

In [1]:
def greet(name):
    return f"Привет, {name}!"

# Положили функцию в переменную — без скобок
alias = greet
print(alias("Аня"))    # Привет, Аня!

# Положили в список
operations = [str.upper, str.lower, str.title]
word = "Привет, мир"
for op in operations:
    print(op(word))

Привет, Аня!
ПРИВЕТ, МИР
привет, мир
Привет, Мир


Функцию-аргумент принимают встроенные `map`, `filter`, `sorted`. Это называется «функция высшего порядка»: на вход или на выход — другая функция.

In [2]:
numbers = [1, 2, 3, 4, 5]

# map применяет функцию к каждому элементу
squared = list(map(lambda x: x ** 2, numbers))
print(squared)              # [1, 4, 9, 16, 25]

# filter оставляет только те, для которых функция вернула True
evens = list(filter(lambda x: x % 2 == 0, numbers))
print(evens)                # [2, 4]

[1, 4, 9, 16, 25]
[2, 4]


## Часть 2. Замыкание — функция внутри функции

Сейчас посмотрим: внутренняя функция запоминает переменные внешней функции, в которой была создана. После завершения внешней функции переменные **не исчезают** — они живут в замыкании.

In [3]:
def make_multiplier(factor):
    def multiply(x):
        return x * factor   # factor — переменная из внешней функции
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(10))    # 20
print(triple(10))    # 30  — каждый замкнул свой factor

20
30


Шаблон «функция, создающая функцию» называется factory function. Часто используется для счётчиков, аккумуляторов, частично применённых функций.

In [4]:
def make_counter():
    count = 0
    def increment():
        nonlocal count       # без nonlocal — UnboundLocalError
        count += 1
        return count
    return increment

counter = make_counter()
print(counter())    # 1
print(counter())    # 2
print(counter())    # 3

1
2
3


## Часть 3. Подводный камень: late-binding в цикле

А что если создать список функций в цикле, замкнув на индекс цикла? Интуиция говорит «каждая функция запомнит свой `i`», а реальность — все функции запоминают **переменную** `i`, и читают её значение в момент вызова. К моменту вызова переменная уже равна последнему значению.

In [5]:
callbacks = []
for i in range(3):
    callbacks.append(lambda: i)

# Все три функции вернут одно и то же — значение i после цикла
for cb in callbacks:
    print(cb())    # 2, 2, 2  — а ожидали 0, 1, 2

2
2
2


Чинится передачей значения через дефолт-параметр (он вычисляется в момент объявления функции, а не вызова):

In [6]:
callbacks = []
for i in range(3):
    callbacks.append(lambda i=i: i)   # i=i замораживает текущее i

for cb in callbacks:
    print(cb())    # 0, 1, 2

0
1
2


## Часть 4. Простой декоратор — замер времени

Декоратор — функция, принимающая функцию и возвращающая другую функцию. Обычно возвращаемая функция (wrapper) добавляет поведение **до** или **после** вызова оригинала.

In [7]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} отработала за {elapsed:.4f} сек")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

result = slow_sum(1_000_000)
print(f"sum = {result}")

slow_sum отработала за 0.0107 сек
sum = 499999500000


Синтаксис `@timer` над `def slow_sum` эквивалентен `slow_sum = timer(slow_sum)`. Декоратор выполняется один раз — в момент объявления функции.

## Часть 5. Проблема: декоратор «съедает» имя и docstring

А что если посмотреть на `__name__` и `__doc__` обёрнутой функции? Wrapper подменил оригинал, поэтому `slow_sum.__name__` — это `wrapper`. Это ломает отладку, тесты, автогенерацию документации.

In [8]:
print(f"имя:    {slow_sum.__name__}")
print(f"doc:    {slow_sum.__doc__}")
print(f"модуль: {slow_sum.__module__}")

имя:    wrapper
doc:    None
модуль: __main__


Чинится декоратором `functools.wraps` — он переносит `__name__`, `__doc__`, `__module__` оригинала на wrapper. Привычка: писать `@wraps` над каждым wrapper.

In [9]:
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} отработала за {elapsed:.4f} сек")
        return result
    return wrapper

@timer
def slow_sum(n):
    """Суммирует числа от 0 до n-1."""
    return sum(range(n))

slow_sum(100_000)
print(f"имя: {slow_sum.__name__}")     # slow_sum, не wrapper
print(f"doc: {slow_sum.__doc__}")      # docstring сохранился

slow_sum отработала за 0.0008 сек
имя: slow_sum
doc: Суммирует числа от 0 до n-1.


## Часть 6. Декоратор с аргументами

Если декоратору самому нужны параметры (например, число повторов или уровень логирования), появляется **третий** уровень. Внешняя функция принимает аргумент, возвращает декоратор, который возвращает wrapper.

In [10]:
def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for i in range(times):
                print(f"повтор {i + 1}/{times}")
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(times=3)
def say_hello(name):
    print(f"  Привет, {name}!")

say_hello("Аня")

повтор 1/3
  Привет, Аня!
повтор 2/3
  Привет, Аня!
повтор 3/3
  Привет, Аня!


## Часть 7. `@lru_cache` — готовый декоратор из стандартной библиотеки

`functools.lru_cache` кэширует результаты вызовов функции по её аргументам. При повторном вызове с теми же аргументами возвращает запомненный результат, не пересчитывая. Работает только с хэшируемыми аргументами (числа, строки, кортежи — да; списки, словари — нет).

In [11]:
from functools import lru_cache

@lru_cache(maxsize=128)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

# Без кэша fib(35) считался бы секунды — с кэшем мгновенно
print(fib(35))         # 9227465
print(fib.cache_info())   # статистика hits / misses

9227465
CacheInfo(hits=33, misses=36, maxsize=128, currsize=36)


## Мини-задания

Три коротких упражнения. Подсказок к именам и методам нет — вспомни сам.

**Задание 1.** Напиши функцию-фабрику `make_adder(n)`, которая возвращает функцию, прибавляющую `n` к своему аргументу. Проверь: `add5 = make_adder(5); add5(10)` должно вернуть `15`.

**Задание 2.** Напиши декоратор `count_calls`, который запоминает, сколько раз была вызвана функция. После каждого вызова он печатает номер этого вызова. Подсказка: счётчик храни в замыкании (нужен `nonlocal`).

**Задание 3.** Что напечатает код ниже? Сначала угадай, потом запусти.

In [12]:
# Задание 1
# def make_adder(...):
#     ...

# Проверка:
# add5 = make_adder(5)
# print(add5(10))   # 15
# print(add5(20))   # 25


In [13]:
# Задание 2
# def count_calls(func):
#     ...

# Проверка:
# @count_calls
# def hello():
#     print('hello')
# hello()    # вызов 1; hello
# hello()    # вызов 2; hello


In [14]:
# Задание 3 — твой прогноз для каждой строки впиши в комментарий:
def make_funcs():
    funcs = []
    for i in range(3):
        funcs.append(lambda: i)
    return funcs

for f in make_funcs():
    print(f())   # ?


2
2
2
